# Step 7 — Match on meaning, not words

*Step 7 of the AI in Industry lab*

---

## Read this before you run anything

You will swap the word-matching retriever for one that matches on meaning, and compare what each one finds.

**What you should end up understanding:** What embeddings are for — and that swapping in a fancier technique does not automatically make a system better.

| | |
|---|---|
| **Cost** | 1 API call |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes. The first run is slower because it loads a small model. |

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b>Uses about 1 API call.</b> Re-running cells is fine, it just uses a little more of your free quota each time.</div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; Honest warning: on this corpus embeddings do NOT clearly beat the simpler method. That is a real result, not a broken notebook. It is exactly why step 11 exists.</b></div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

Step 6(c) showed word-matching failing on *"exemption"* versus *"condoned"*.

**Embeddings** turn text into a list of numbers, positioned so that similar meanings land near each other. A match no longer needs shared words.

The vectors for all 227 clauses ship with the lab, already computed.

In [ ]:
from labcore import corpus, tfidf, embed, grounded

chunks, texts, _ = corpus()
by_word    = tfidf(texts)      # step 4's retriever
by_meaning = embed(texts)      # the new one

print("both retrievers ready")

### The paraphrased question

The one that broke word-matching.

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = "Can I get an exemption if I miss too many classes?"

### Compare what each retriever finds

In [ ]:
print("BY WORD:")
for t, s in by_word(question, k=2):
    print(f"   {s:.3f}  {t[:80]}...")

print()
print("BY MEANING:")
for t, s in by_meaning(question, k=2):
    print(f"   {s:.3f}  {t[:80]}...")

# The two score scales are different. What matters is WHICH chunks
# come back, and in what order - not the numbers themselves.

## An honest result

**You may find embeddings do not clearly win here.** That is not a broken notebook — it is a real measurement.

Six different phrasings were tested across two embedding models. Neither reliably beat the simpler method on this corpus, and sometimes they did worse. The right clause *is* in there; it just ranks below other attendance clauses that look equally plausible to both.

This is the honest state of the field. A newer, fancier technique is not automatically better on **your** data, and you cannot tell from one example.

That is exactly why step 11 exists.

---

## Ask, using the new retriever

In [ ]:
print(grounded(by_meaning)(question, k=4))

---

## Now change it yourself

Hunt for a question where the two genuinely disagree. Change the question cell above and re-run the comparison.

Worth trying:

- `"What if I am sick and cannot attend for two weeks?"`
- `"Can I be let off for poor attendance?"`
- `"I was representing the college at a sports event"`

Keep a note of any where meaning-matching clearly wins. You will need evidence like that to justify the extra complexity to anyone.

---

### Done with step 7

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.